In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [0]:
import os, sys, json, re
import pandas as pd

# point to working_branch src
sys.path.append("/Workspace/9900-f18a-cake/working_branch/src")
print("sys.path appended.")

sys.path appended.


In [0]:
# Databricks notebook source
import joblib
import pandas as pd

TREE_PATH = "/Workspace/9900-f18a-cake/working_branch/data/freeze0525/diseaseTree_mapped.joblib"
tree = joblib.load(TREE_PATH)

nodes = []

def node_label(node):
    for attr in ["display_name", "label", "class_name", "disease", "name", "id"]:
        if hasattr(node, attr):
            v = getattr(node, attr)
            if v is not None and str(v).strip() != "":
                return str(v)
    return str(node)

def dfs(node, path=()):
    label = node_label(node)
    cur_path = path + (label,)

    # direct samples
    samples = getattr(node, "samples", None) or []
    n_direct = len(samples)

    # children
    children = getattr(node, "children", None) or []
    n_subtree = n_direct
    for c in children:
        n_subtree += dfs(c, cur_path)

    nodes.append({
        "node": label,                      # may dup
        "path": " / ".join(cur_path),       
        "n_direct": n_direct,               # only this node's sample
        "n_samples": n_subtree,             # plus child sample
    })
    return n_subtree

dfs(tree)

df = (
    pd.DataFrame(nodes)
      .sort_values("n_samples", ascending=False)
      .reset_index(drop=True)
)

display(df)
print("Total nodes:", len(df))


node,path,n_direct,n_samples
ZERO2,ZERO2,0,1868
Central nervous system,ZERO2 / Central nervous system,0,633
Haematological malignancy,ZERO2 / Haematological malignancy,0,496
Leukaemia,ZERO2 / Haematological malignancy / Leukaemia,0,420
Sarcoma,ZERO2 / Sarcoma,0,402
B-lymphoblastic leukaemia,ZERO2 / Haematological malignancy / Leukaemia / B-lymphoblastic leukaemia,239,239
Solid tumour,ZERO2 / Solid tumour,0,215
Paediatric-type diffuse glioma,ZERO2 / Central nervous system / Paediatric-type diffuse glioma,0,186
CNS embryonal tumour,ZERO2 / Central nervous system / CNS embryonal tumour,0,141
Neuroblastoma,ZERO2 / Neuroblastoma,117,119


Total nodes: 296


In [0]:
# df_gt50 = df[df["n_samples"] > 50].sort_values("n_samples", ascending=False)
# display(df_gt50)

# nodes_gt50 = df_gt50["path"].tolist()  


In [0]:
TEST_NODE = ["Haematological malignancy", "Sarcoma"]
df_gt50 = df[df["node"].isin(TEST_NODE)].copy()
nodes_gt50 = df_gt50["node"].tolist()      # only train ['Haematological malignancy','Sarcoma']
display(df_gt50)

node,path,n_direct,n_samples
Haematological malignancy,ZERO2 / Haematological malignancy,0,496
Sarcoma,ZERO2 / Sarcoma,0,402


In [0]:
SMALL_MAX = 80
MID_MAX   = 500

def bucket(n):
    if n < SMALL_MAX:
        return "small"
    if n < MID_MAX:
        return "mid"
    return "large"

df_gt50["bucket"] = df_gt50["n_samples"].apply(bucket)

small_nodes = df_gt50[df_gt50.bucket=="small"]["node"].tolist()
mid_nodes   = df_gt50[df_gt50.bucket=="mid"]["node"].tolist()
large_nodes = df_gt50[df_gt50.bucket=="large"]["node"].tolist()

print("SMALL:", small_nodes)
print("MID:", mid_nodes)
print("LARGE:", large_nodes)
display(df_gt50)


SMALL: []
MID: ['Haematological malignancy', 'Sarcoma']
LARGE: []


node,path,n_direct,n_samples,bucket
Haematological malignancy,ZERO2 / Haematological malignancy,0,496,mid
Sarcoma,ZERO2 / Sarcoma,0,402,mid


In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# job id
MANUAL_JOB_ID = 1065227880999896

# child notebook directory
CHILD_NB_PATH = "/Workspace/9900-f18a-cake/working_branch/Training_model_child"

def safe_key(s):
    return s.replace(" ", "_").replace("/", "_")[:90]

def make_key(prefix, idx):
    return f"{prefix}_{idx:03d}"

# assign tasks dynamically
tasks = []
i = 0

for n in small_nodes:
    tasks.append({
        "task_key": make_key("S", i),
        "job_cluster_key": "jobClustertest_S",
        "notebook_task": {
            "notebook_path": CHILD_NB_PATH,
            "base_parameters": {"NODE_ID": n},
        }
    })
    i += 1

for n in mid_nodes:
    tasks.append({
        "task_key": make_key("M", i),
        "job_cluster_key": "jobClustertest_M",
        "notebook_task": {
            "notebook_path": CHILD_NB_PATH,
            "base_parameters": {"only_node": n},
        }
    })
    i += 1

for n in large_nodes:
    tasks.append({
        "task_key": make_key("L", i),
        "job_cluster_key": "jobClustertest_L",
        "notebook_task": {
            "notebook_path": CHILD_NB_PATH,
            "base_parameters": {"only_node": n},
        }
    })
    i += 1

print("Prepared tasks:", len(tasks))
print(tasks[:1]) 


Prepared tasks: 2
[{'task_key': 'M_000', 'job_cluster_key': 'jobClustertest_M', 'notebook_task': {'notebook_path': '/Workspace/9900-f18a-cake/working_branch/Training_model_child', 'base_parameters': {'only_node': 'Haematological malignancy'}}}]


In [0]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

#manual insert job id
job_id_S = 628462771387153
job_id_M = 1114821904435730
job_id_L = 959751840322526

# merge S/M/L
all_tasks = []
for node in small_nodes:
    all_tasks.append(("S", node))
for node in mid_nodes:
    all_tasks.append(("M", node))
for node in large_nodes:
    all_tasks.append(("L", node))

print("Total tasks:", len(all_tasks))

def submit_run(job_type, node):
    if job_type == "S":
        job_id = job_id_S
    elif job_type == "M":
        job_id = job_id_M
    else:
        job_id = job_id_L

    run = w.jobs.run_now(
        job_id=job_id,
        notebook_params={"only_node": node}
    )
    print(f"Submitted: {node} -> run_id={run.run_id}")
    return run.run_id

def wait_for_runs(run_ids):
    """
    waiting current batch end
    """
    print(f"Waiting for runs: {run_ids}")
    TERMINAL_LIFE   = {"TERMINATED", "SKIPPED", "INTERNAL_ERROR"}
    TERMINAL_RESULT = {"SUCCESS", "FAILED", "TIMEDOUT", "CANCELED"}

    while True:
        finished = 0

        for rid in run_ids:
            run = w.jobs.get_run(run_id=rid)
            state = run.state

            life   = str(state.life_cycle_state)
            result = str(state.result_state)

            print(f"run {rid}: life={life}, result={result}")


            if any(x in life for x in TERMINAL_LIFE):
                finished += 1

        if finished == len(run_ids):
            print(f"Batch finished: {run_ids}")
            return

        time.sleep(15)

# ---- run in different batch ----
batch = []
for job_type, node in all_tasks:
    run_id = submit_run(job_type, node)
    batch.append(run_id)

    # if there are 3 running, wait
    if len(batch) == 3:
        wait_for_runs(batch)
        batch = []

# last batch need to wait if there is not 3 running
if batch:
    wait_for_runs(batch)

print("All runs completed!")


Total tasks: 2
Submitted: Haematological malignancy -> run_id=989814493366142
Submitted: Sarcoma -> run_id=553334815108361
Waiting for runs: [989814493366142, 553334815108361]
run 989814493366142: life=RunLifeCycleState.RUNNING, result=None
run 553334815108361: life=RunLifeCycleState.RUNNING, result=None
run 989814493366142: life=RunLifeCycleState.RUNNING, result=None
run 553334815108361: life=RunLifeCycleState.RUNNING, result=None
run 989814493366142: life=RunLifeCycleState.RUNNING, result=None
run 553334815108361: life=RunLifeCycleState.RUNNING, result=None
run 989814493366142: life=RunLifeCycleState.RUNNING, result=None
run 553334815108361: life=RunLifeCycleState.RUNNING, result=None
run 989814493366142: life=RunLifeCycleState.RUNNING, result=None
run 553334815108361: life=RunLifeCycleState.RUNNING, result=None
run 989814493366142: life=RunLifeCycleState.RUNNING, result=None
run 553334815108361: life=RunLifeCycleState.RUNNING, result=None
run 989814493366142: life=RunLifeCycleState.